# EVA Cesar · 02. Limpieza y agregación de EVA Cesar

Programación en Lenguajes Estadísticos · UNAL

## Montaje de Drive y ubicación del proyecto

Ejecutar desde la raíz del proyecto o indicar `EVA_RAIZ`. Correr los cuadernos en orden 01, 02 y 03.

In [ ]:
from pathlib import Path
import os

ruta_local = Path(os.environ.get("EVA_RAIZ", ".")).resolve()
try:
    from google.colab import drive
except ImportError:
    raiz = ruta_local
else:
    drive.mount("/content/drive")
    rutas = [ruta_local, Path("/content/drive/MyDrive/proyecto_eva_cesar"),
             Path("/content/drive/Shareddrives/Programacion en lenguajes estadisticos/proyecto_eva_cesar")]
    raiz = next((p for p in rutas if (p / "datos/datos_crudos").is_dir()), rutas[1])
if not (raiz / "datos/datos_crudos").is_dir():
    raise FileNotFoundError(f"Indique la carpeta raíz de EVA con EVA_RAIZ. Ruta revisada: {raiz}")
docs = raiz / "documentos"
limpios = raiz / "datos/datos_limpios"
figuras = docs / "figuras"
for carpeta in (docs, limpios, figuras):
    carpeta.mkdir(parents=True, exist_ok=True)


## Normalización y comprobaciones

In [ ]:
import numpy as np
import pandas as pd

def exportar(tabla, carpeta, nombre):
    tabla.to_csv(carpeta / nombre, index=False)

eva = pd.read_csv(limpios / "eva_cesar_2006_2018_sin_limpiar_py.csv", dtype={"cod_dep": str, "cod_mun": str})
bitacora = [{"paso": "subconjunto_original", "filas": len(eva)}]
def registrar(paso, tabla):
    bitacora.append({"paso": paso, "filas": len(tabla)})

texto = eva.select_dtypes(include=["object", "string"]).columns.difference(["cod_dep", "cod_mun"])
for columna in texto:
    eva[columna] = eva[columna].str.replace(r"\s+", " ", regex=True).str.strip().str.upper()
eva = eva.drop_duplicates().copy()
registrar("sin_duplicados_exactos", eva)
clave = ["cod_mun", "cultivo", "ano", "periodo", "desagregacion_regional_y_o_sistema_productivo"]
assert not eva[clave].isna().any().any() and not eva.duplicated(clave).any()
assert not eva[["cod_mun", "municipio"]].drop_duplicates()["cod_mun"].duplicated().any()
medidas = ["area_sembrada_ha", "area_cosechada_ha", "produccion_t"]
valido = (np.isfinite(eva[medidas]).all(axis=1) & eva[medidas].ge(0).all(axis=1)
          & eva["area_cosechada_ha"].le(eva["area_sembrada_ha"])
          & (eva["area_cosechada_ha"].gt(0) | eva["produccion_t"].eq(0)))
exportar(eva.loc[~valido], docs, "e2_Python_incoherentes.csv")
eva = eva.loc[valido].copy()
registrar("medidas_coherentes", eva)
eva["sin_cosecha"] = eva["area_cosechada_ha"].eq(0)
cociente = eva["produccion_t"] / eva["area_cosechada_ha"].where(~eva["sin_cosecha"])
eva["rend_discrepante"] = (cociente - eva["rendimiento_t_ha"]).abs().gt(0.05 * eva["rendimiento_t_ha"] + 0.01)

## Agregación anual en pandas

Las hectáreas sembradas sin cosecha participan en los totales. Si la cosecha anual es cero, el rendimiento de esa combinación queda indefinido. Usamos 2007–2018 porque 2006 solo contiene el periodo B en Cesar.

In [ ]:
es_anual = eva["ano"].between(2007, 2018)
area_cultivos = (eva.loc[es_anual].groupby("cultivo", as_index=False)["area_sembrada_ha"].sum()
                .sort_values(["area_sembrada_ha", "cultivo"], ascending=[False, True]))
area_cultivos["fraccion_area"] = area_cultivos["area_sembrada_ha"] / area_cultivos["area_sembrada_ha"].sum()
principales = area_cultivos["cultivo"].head(10).tolist()
area_cultivos["principal"] = area_cultivos["cultivo"].isin(principales)
eva["cultivo_principal"] = eva["cultivo"].isin(principales)
filas = eva.loc[es_anual & eva["cultivo_principal"]].copy()
registrar("2007_2018_cultivos_principales", filas)
grupos = ["cod_mun", "municipio", "cultivo", "ano"]
agrupado = filas.groupby(grupos, as_index=False)
anual = agrupado.agg(n_registros=("cultivo", "size"), area_sembrada_ha=("area_sembrada_ha", "sum"),
                    area_cosechada_ha=("area_cosechada_ha", "sum"), produccion_t=("produccion_t", "sum"),
                    n_discrepancias=("rend_discrepante", "sum"), n_sin_cosecha=("sin_cosecha", "sum"))
assert not anual.duplicated(["cod_mun", "cultivo", "ano"]).any()
anual["rendimiento_t_ha"] = anual["produccion_t"] / anual["area_cosechada_ha"].where(anual["area_cosechada_ha"].gt(0))
anual["log_rendimiento"] = np.log(anual["rendimiento_t_ha"].where(anual["rendimiento_t_ha"].gt(0)))

## Atípicos con el rango intercuartílico
          
Trabajamos con el logaritmo natural de los rendimientos positivos. En cada cultivo, el rango intercuartílico es Q3 − Q1. Marcamos los valores por debajo de Q1 − 3 × IQR o por encima de Q3 + 3 × IQR. Las cercas exteriores reservan la exclusión para extremos. Las dos versiones de la tabla permiten comprobar si esos casos cambian las conclusiones.

In [ ]:
anual["atipico_severo"] = False
for columna in ["cerca_inf_log", "cerca_sup_log"]:
    anual[columna] = np.nan
for cultivo in principales:
    x = anual.loc[anual["cultivo"].eq(cultivo), "log_rendimiento"].dropna()
    if len(x) < 20:
        continue
    q1, q3 = x.quantile([0.25, 0.75], interpolation="linear")
    amplitud = q3 - q1
    if amplitud == 0:
        continue
    inferior, superior = q1 - 3 * amplitud, q3 + 3 * amplitud
    anual.loc[x.index, "atipico_severo"] = (x < inferior) | (x > superior)
    anual.loc[x.index, "cerca_inf_log"] = inferior
    anual.loc[x.index, "cerca_sup_log"] = superior

depurada = anual.loc[~anual["atipico_severo"]]
candidatos = anual.loc[anual["atipico_severo"]].assign(motivo="fuera de las cercas exteriores de 3 IQR")
origen = candidatos[grupos].merge(eva, on=grupos, validate="one_to_many")

def describir(x):
    q = x.quantile([0, .25, .5, .75, 1])
    return dict(n=x.count(), minimo=q[0], q1=q[.25], mediana=q[.5], q3=q[.75], maximo=q[1],
                media=x.mean(), de=x.std(), iqr=q[.75] - q[.25])

descriptivos = pd.DataFrame([dict(variable=v, **describir(anual[v])) for v in medidas + ["rendimiento_t_ha"]])
por_cultivo = pd.DataFrame([dict(cultivo=k, base=base, **describir(d.loc[d["cultivo"].eq(k), "rendimiento_t_ha"]))
                           for k in principales for base, d in [("completa", anual), ("depurada", depurada)]])

for tabla, nombre in [(eva, "limpio"), (anual, "anual"), (depurada, "anual_depurada")]:
    exportar(tabla, limpios, f"eva_cesar_{nombre}_py.csv")
for tabla, nombre in [(area_cultivos, "cobertura_cultivos"), (pd.DataFrame(bitacora), "bitacora_limpieza"),
                      (candidatos, "atipicos_anuales"), (origen, "atipicos_origen"),
                      (descriptivos, "descriptivos_generales"), (por_cultivo, "descriptivos_por_cultivo")]:
    exportar(tabla, docs, f"e2_Python_{nombre}.csv")
display(pd.DataFrame(bitacora), candidatos, por_cultivo)
print(f"Filas anuales: {len(anual)} | sin cosecha: {anual.rendimiento_t_ha.isna().sum()} | extremos: {len(candidatos)}")